# NV3 — Baseline phát hiện ngã liên tục (notebook tái lập, chạy được trên Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ank-less/ABC-UWB-Falltrack/blob/sangkhoi/notebooks/02_baseline_colab.ipynb)

Notebook chạy lại toàn bộ pipeline baseline:
tải dữ liệu → kiểm tra → split chính thức + CV theo người → đặc trưng → mô hình → chọn ngưỡng trên train → đánh giá val bằng evaluator chính thức.

- `SMOKE = True`: 2 luồng mỗi split, mỗi luồng 300 s, tải khoảng 1,1 GB, chạy vài phút. Dùng để kiểm tra pipeline.
- `SMOKE = False`: 13 luồng train và 5 luồng val, tải khoảng 4,6 GB, chạy khoảng 5–10 phút. Đây là cách tạo ra kết quả trong [`reports/baseline_results_report.md`](../reports/baseline_results_report.md).

Mọi tham số nằm trong `configs/baseline.yaml`. Seed cố định là 2027. Mỗi lần chạy tạo một thư mục mới trong `outputs/experiments/`.

In [ ]:
SMOKE = True   # đổi thành False để chạy đầy đủ

In [ ]:
# Chạy được cả trên Colab lẫn máy cá nhân.
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/ank-less/ABC-UWB-Falltrack.git"
BRANCH = "sangkhoi"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not Path("ABC-UWB-Falltrack").exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL], check=True)
    os.chdir("ABC-UWB-Falltrack")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[analysis,test]"], check=True)
else:
    # notebook nằm trong notebooks/: chuyển về thư mục gốc của repo
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent
    os.chdir(root)
if str(Path.cwd() / "src") not in sys.path:
    sys.path.insert(0, str(Path.cwd() / "src"))
print("working directory:", Path.cwd())

## 1. Tải dữ liệu từ link chính thức

`scripts/fetch_data.py` chỉ rút những file cần thiết từ `abc2027_uwb_train_v1.zip` bằng HTTP range request, rồi kiểm tra CRC32 của từng file. Thư mục `segments/` không được tải.

In [ ]:
subprocess.run([sys.executable, "scripts/fetch_data.py", "--subset", "smoke" if SMOKE else "all"], check=True)

## 2. Kiểm tra tự động (dùng dữ liệu tổng hợp)

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"], capture_output=True, text=True)
print(result.stdout[-1500:])

## 3. Chạy pipeline

In [ ]:
command = [sys.executable, "scripts/run_baseline.py", "--config", "configs/baseline.yaml", "--stage", "all"]
if SMOKE:
    command.append("--smoke")
result = subprocess.run(command, capture_output=True, text=True)
print(result.stderr[-3000:])
print(result.stdout)
result.check_returncode()
run_dir = Path(result.stdout.strip().split("run folder: ")[-1])
run_dir

## 4. Kết quả

- **train OOF**: điểm out-of-fold trên train, với các fold chia theo người. Đây là con số duy nhất được dùng để chọn mô hình và ngưỡng.
- **val**: tập holdout chính thức, chấm bằng `evaluate.py` của ban tổ chức với `--ignore-flagged`.
- `f1_mean` là điểm xếp hạng chính thức: trung bình F1 sự kiện tại θ = 0,1 / 0,3 / 0,5.

In [ ]:
import json
import pandas as pd
final = json.loads((run_dir / "final_metrics.json").read_text())
rows = []
for name, entry in final["models"].items():
    by_theta = entry["holdout"]["event"]["by_theta"]
    rows.append({
        "model": name,
        "selected": name == final["selected_model"],
        "train_in_sample_f1_mean": entry["train_insample"]["event_f1_mean"],
        "train_oof_f1_mean": entry["train_oof"]["event_f1_mean"],
        "val_f1_mean": entry["holdout"]["event"]["ranking_score"]["f1_mean"],
        **{f"val_f1@{t}": by_theta[t]["f1"] for t in ("0.1", "0.3", "0.5", "0.7")},
        "val_fa_per_h@0.3": by_theta["0.3"]["false_alarms_per_hour"],
        "val_window_pr_auc": entry["holdout"]["window"]["pr_auc"],
    })
pd.DataFrame(rows).round(3)

In [ ]:
for name, entry in final["models"].items():
    print(name)
    for note in entry["diagnostics"]:
        print("  -", note)
selection = json.loads((run_dir / "selection.json").read_text())
pd.DataFrame({m: {**v["postprocessing"], "oof_f1_mean": v["oof_f1_mean"], **v["model_params"]} for m, v in selection["models"].items()})

In [ ]:
import matplotlib.pyplot as plt
selected = final["selected_model"]
sens = pd.read_csv(run_dir / "threshold_sensitivity.csv")
pr = pd.read_csv(run_dir / f"pr_curve_oof_{selected}.csv")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for name, group in sens.groupby("model"):
    axes[0].plot(group["threshold"], group["f1_mean"], marker="o", label=name)
axes[0].set_xlabel("threshold"); axes[0].set_ylabel("train OOF f1_mean"); axes[0].legend(); axes[0].set_title("Threshold sensitivity (train OOF)")
axes[1].plot(pr["recall"], pr["precision"]); axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision")
axes[1].set_title(f"Window PR curve, train OOF ({selected})")
plt.tight_layout(); plt.show()

In [ ]:
# Một đoạn của luồng val: score của cửa sổ, sự kiện dự đoán và cú ngã thật
windows = pd.read_csv(run_dir / "window_predictions_val.csv.gz")
events = pd.read_csv("data/uwb_falltrack/abc2027_uwb_train_v1/streams_events.csv")
pred = pd.read_csv(run_dir / "submission_val.csv")
stream, t0, t1 = "val_00", 0, 240
w = windows[(windows.stream_id == stream) & windows.t_start.between(t0, t1)]
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(0.5 * (w.t_start + w.t_end), w[f"score_{selected}"], lw=0.8, label="window score")
for e in events[(events.stream_id == stream) & (events.t_start < t1)].itertuples():
    ax.axvspan(e.t_start, e.t_end, color="red", alpha=0.3)
for p in pred[(pred.stream_id == stream) & (pred.t_start < t1)].itertuples():
    ax.axvspan(p.t_start, p.t_end, ymin=0.9, ymax=1.0, color="black")
thr = selection["models"][selected]["postprocessing"]["threshold"]
ax.axhline(thr, color="grey", ls="--", label=f"threshold {thr}")
ax.set_xlim(t0, t1); ax.set_xlabel("time (s)"); ax.legend(loc="upper right")
ax.set_title(f"{stream}: red = annotated fall, black bars = predicted events")
plt.show()

## 5. Chấm lại file submission bằng evaluator chính thức

Chỉ có ý nghĩa với run đầy đủ (`SMOKE = False`). Ở chế độ smoke, các luồng bị cắt còn 300 s, trong khi bước này đọc độ dài đầy đủ của luồng val.

In [ ]:
out = run_dir / "submission_val.report.json"
if SMOKE:
    print("smoke run: skipped (truncated streams)")
else:
    if not out.exists():
        subprocess.run([sys.executable, "scripts/evaluate_predictions.py", "--config", "configs/baseline.yaml",
                        "--predictions", str(run_dir / "submission_val.csv"), "--split", "val"], check=True)
    print(json.loads(out.read_text())["ranking_score"])

## 6. Tái lập

Thư mục run chứa đủ thông tin để chạy lại: `resolved_config.yaml`, `environment.json` (phiên bản thư viện và git commit), `split_assignments.csv`, `selection.json`, model `.joblib`, các bảng metric và file dự đoán.